# IGCA on Falcon-40B: zero-gated cross-attention, end to end (v1.1)

Reference notebook for **Zero-Gated Cross-Attention** ([arXiv:2611.01116](https://arxiv.org/abs/2611.01116)).
Frozen Falcon decoder (4-bit NF4) + frozen CLIP ViT-L/14 + six IGCA bridges at layers
`[3, 7, 11, 15, 19, 23]`, trained on Flickr8k captioning and evaluated on the **disjoint
validation split** with the gate on (IGCA) and off (frozen backbone).

**What this notebook guarantees**
* The zero-gate identity is checked to be **bit-exact (0.00e+00)** before training starts.
* The model that actually runs is the model that is reported: there is **no silent fallback**.
* Every number is written to `run_manifest.json`, `loss_log.csv` and `eval_log.csv`.

**Hardware.** Falcon-40B needs roughly 85 GB of disk for the download and about 25 GB of GPU
memory in 4-bit (for example one A100-40GB or 2× A6000). Kaggle's free T4×2 has too little disk
for Falcon-40B; there, set `BASE_MODEL_ID = 'tiiuae/falcon-7b'` explicitly.

*v1.1 changes vs the notebook released with arXiv v1 are listed in `ERRATA.md`.*

In [ ]:
# Must run before torch touches CUDA.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
OUT_ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

# transformers 4.48.x + accelerate 1.4.x: tested 4-bit path for Falcon.
!pip install -q "transformers==4.48.3" "accelerate==1.4.0" "datasets" "pillow==11.3.0" "huggingface_hub" "bitsandbytes"

import torch
print('CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}: {p.name} | {p.total_memory/1e9:.1f} GB')
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Enable a GPU accelerator, or run `python falcon_local.py --tiny-random` on CPU.')

In [ ]:
# Fetch the reference implementation (igca.py, igca_pipeline.py) if not already present.
import sys, subprocess
REPO_URL = 'https://github.com/Basavarajearlybird/IGCA-Zero-Gated-Cross-Attention-'
if not os.path.exists('igca.py'):
    if not os.path.isdir('igca_repo'):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, 'igca_repo'], check=True)
    sys.path.insert(0, os.path.abspath('igca_repo'))
import igca
import igca_pipeline as pipe
print('IGCA reference implementation', igca.__version__)

In [ ]:
# ================= CONFIG (edit here) =================
BASE_MODEL_ID     = 'tiiuae/falcon-40b'
VISION_ENCODER_ID = 'openai/clip-vit-large-patch14'
VARIANT           = 'gated'      # 'gated' = IGCA (Eq. 2) | 'ungated' = standard adapter baseline (Eq. 1)

STRIDE, MAX_MODULES, N_HEADS = 4, 6, 8   # -> layers [3, 7, 11, 15, 19, 23] on a 60-layer decoder
BATCH_SIZE, SEQLEN           = 1, 48
MAX_STEPS                    = 59_500    # optimizer updates (independent of dataset size)
TRAIN_ROWS                   = None      # None = all 6,000 Flickr8k training captions
HELDOUT_SPLIT, HELDOUT_N     = 'validation', 200   # disjoint from training
EVAL_EVERY                   = 500
LR_GATE, LR_WEIGHT           = 1e-3, 5e-5
OBJECTIVE, L2_MU             = 'adamw', 0.0    # 'alg1' adds mu*||theta||^2 exactly as in Algorithm 1
SEED                         = 0
ALLOW_CPU_OFFLOAD            = False

torch.manual_seed(SEED)
print(f'{BASE_MODEL_ID} | {VARIANT} | stride {STRIDE} x {MAX_MODULES} | '
      f'{MAX_STEPS} steps | batch {BATCH_SIZE} | seqlen {SEQLEN}')

In [ ]:
from PIL import Image
import transformers, bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel, AutoImageProcessor, BitsAndBytesConfig
assert transformers.__version__.startswith('4.'), 'transformers 5.x ignores bnb 4-bit here: restart after the pinned install.'
print('torch', torch.__version__, '| transformers', transformers.__version__, '| bitsandbytes', bnb.__version__)

## Load the frozen backbone (4-bit NF4), explicitly

A pre-flight check compares the download size with the free disk and the 4-bit footprint
with GPU memory **before** downloading. If the configured model does not fit, the notebook
**stops** with a clear message instead of switching models.

In [ ]:
import gc, shutil
from collections import Counter

def hf_download_gb(model_id):
    try:
        from huggingface_hub import HfApi
        info = HfApi().model_info(model_id, files_metadata=True)
        total = sum((s.size or 0) for s in info.siblings if s.rfilename.endswith(('.safetensors', '.bin')))
        if total:
            return total / 1e9
    except Exception:
        pass
    return {'tiiuae/falcon-40b': 85.0, 'tiiuae/falcon-7b': 15.0}.get(model_id, 20.0)

dl_gb = hf_download_gb(BASE_MODEL_ID)
hf_home = os.environ.get('HF_HOME', os.path.expanduser('~/.cache/huggingface'))
os.makedirs(hf_home, exist_ok=True)
free_gb = shutil.disk_usage(hf_home).free / 1e9
n_gpu = torch.cuda.device_count()
gpu_total = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpu)) / 1e9
need_gpu = 0.55 * (dl_gb / 2) + 2.0
print(f'pre-flight {BASE_MODEL_ID}: download ~{dl_gb:.0f} GB (disk free {free_gb:.0f} GB) | '
      f'4-bit need ~{need_gpu:.0f} GB (GPU total {gpu_total:.0f} GB)')
if dl_gb * 1.15 > free_gb or need_gpu > gpu_total:
    raise SystemExit(f'{BASE_MODEL_ID} does not fit this machine. Change BASE_MODEL_ID explicitly '
                     '(e.g. tiiuae/falcon-7b); the manifest records whichever model runs.')

largest = max(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpu)) / 1e9
DEVICE_MAP = {'': 0} if need_gpu < 0.9 * largest else 'auto'     # single-GPU pin avoids cross-device ops
max_memory = {i: int(torch.cuda.get_device_properties(i).total_memory * 0.90) for i in range(n_gpu)}
if ALLOW_CPU_OFFLOAD:
    max_memory['cpu'] = '48GiB'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load(quant):
    cfg = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16,
                              llm_int8_enable_fp32_cpu_offload=ALLOW_CPU_OFFLOAD)
           if quant == 'nf4-4bit' else
           BitsAndBytesConfig(load_in_8bit=True, llm_int8_enable_fp32_cpu_offload=ALLOW_CPU_OFFLOAD))
    return AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=cfg, device_map=DEVICE_MAP,
                                                max_memory=max_memory, torch_dtype=torch.float16)

QUANT = 'nf4-4bit'
llm = load(QUANT)
dtypes = Counter(str(p.dtype) for p in llm.parameters())
if 'torch.uint8' not in dtypes:            # 4-bit did not apply in this environment
    print('4-bit quantisation did not apply; reloading the SAME model in 8-bit (recorded in the manifest).')
    del llm; gc.collect(); torch.cuda.empty_cache()
    QUANT = 'int8'
    llm = load(QUANT)
FROZEN = igca.freeze(llm)
try:
    llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
except TypeError:
    llm.gradient_checkpointing_enable()
llm.enable_input_require_grads()
llm.config.use_cache = False
llm.train()

vision_encoder = AutoModel.from_pretrained(VISION_ENCODER_ID, torch_dtype=torch.float16).to('cuda:0').eval()
igca.freeze(vision_encoder)
image_processor = AutoImageProcessor.from_pretrained(VISION_ENCODER_ID)
MODALITY_DIM = vision_encoder.config.vision_config.hidden_size
print(f'Loaded {BASE_MODEL_ID} ({QUANT}) | footprint {llm.get_memory_footprint()/1e9:.1f} GB | '
      f'hidden {llm.config.hidden_size} | layers {llm.config.num_hidden_layers} | modality dim {MODALITY_DIM}')

## Attach IGCA and count trainable parameters

Each bridge is placed on the device of the layer it hooks. The measured parameter count is
asserted against the closed form `6 × (2H² + 2HD + 4H + 1)`; for Falcon-40B
(H = 8192, D = 1024) that is **906,166,278**.

In [ ]:
bridge = igca.IGCABridge(llm, MODALITY_DIM, stride=STRIDE, max_modules=MAX_MODULES, n_heads=N_HEADS,
                         gated=(VARIANT == 'gated'), dtype=torch.float16)
N_TRAINABLE = bridge.num_trainable()
assert N_TRAINABLE == bridge.expected_trainable()
print(f'decoder path {bridge.layer_path} | IGCA layers {bridge.indices}')
print(f'trainable IGCA params: {N_TRAINABLE:,} (~{N_TRAINABLE/1e9:.3f}B) | frozen backbone params: {FROZEN:,}')

## Step-0 identity check (hard gate)

With `alpha = 0`, attaching IGCA must leave every logit **bit-identical**. The un-gated
baseline is reported for contrast: it already perturbs the frozen model at initialisation.

In [ ]:
import numpy as np
dummy = Image.fromarray(np.random.default_rng(0).integers(0, 255, (224, 224, 3), dtype=np.uint8))
px = image_processor(images=dummy, return_tensors='pt').pixel_values
probe = tokenizer('A photo of', return_tensors='pt').to(next(llm.parameters()).device)
STEP0_DIFF = igca.zero_gate_max_diff(llm, bridge, {'input_ids': probe['input_ids']},
                                     pipe.embed_images(vision_encoder, px))
print(f'step-0 max |logit diff| (bridge on vs off): {STEP0_DIFF:.2e}')
if VARIANT == 'gated':
    assert STEP0_DIFF == 0.0, 'Zero-gate identity FAILED: restart the runtime (stale hooks) and re-run.'
    print('Exact no-op confirmed -> safe to train.')

In [ ]:
from torch.utils.data import DataLoader
train_ds, heldout_ds = pipe.load_flickr8k(TRAIN_ROWS, HELDOUT_SPLIT, HELDOUT_N)
enc = pipe.CaptionEncoder(tokenizer, SEQLEN)
loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=lambda ex: enc.collate(ex, image_processor))
train_iter = pipe.infinite(loader)
EPOCHS = MAX_STEPS * BATCH_SIZE / len(train_ds)
print(f'Flickr8k train captions: {len(train_ds)} | held-out ({HELDOUT_SPLIT}): {len(heldout_ds)} | '
      f'{MAX_STEPS} steps = {EPOCHS:.1f} epochs')
if EPOCHS > 3:
    print('Note: many epochs over 6,000 captions -> expect train loss to approach 0 (memorisation); '
          'the held-out loss is the meaningful metric.')

def heldout(gate_on):
    return pipe.heldout_loss(llm, bridge, heldout_ds, enc, vision_encoder, image_processor, gate_on)

## Train, keeping the best held-out checkpoint

In [ ]:
import time
optimizer, OPT_NAME = pipe.build_optimizer(bridge, LR_GATE, LR_WEIGHT, OBJECTIVE)
SLUG = pipe.model_slug(BASE_MODEL_ID)
RUN_DIR = os.path.join(OUT_ROOT, f'igca_{SLUG}_{VARIANT}_{time.strftime("%Y%m%d-%H%M%S")}')
log = pipe.RunLogger(RUN_DIR)
torch.cuda.reset_peak_memory_stats()

OFF = heldout(False)
ON0 = heldout(True)
log.eval(0, ON0, OFF, bridge.mean_gate())
print(f'held-out gate-OFF (frozen {SLUG}): {OFF:.4f} | step 0 with bridge active: {ON0:.4f}')

best = {'loss': float('inf'), 'step': 0, 'gate': 0.0}
loss_history, t0 = [], time.time()
for step in range(1, MAX_STEPS + 1):
    batch = next(train_iter)
    e = pipe.embed_images(vision_encoder, batch['pixel_values'])
    loss = pipe.train_step(llm, bridge, batch, e, optimizer, OBJECTIVE, L2_MU)
    loss_history.append(loss)
    log.step(step, loss, bridge.mean_gate())
    if step == 5:
        sps = (time.time() - t0) / 5
        print(f'measured {sps:.2f} s/step -> projected {sps * MAX_STEPS / 3600:.1f} h for {MAX_STEPS} steps')
    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        on, g = heldout(True), bridge.mean_gate()
        log.eval(step, on, OFF, g)
        print(f'step {step:>6} | train {loss:.4f} | held-out on {on:.4f} (off {OFF:.4f}) | gate {g:.4f}')
        if on < best['loss']:
            best = {'loss': on, 'step': step, 'gate': g}
            torch.save({'igca_state': bridge.modules.state_dict(), 'step': step, 'model_id': BASE_MODEL_ID,
                        'indices': bridge.indices, 'variant': VARIANT},
                       os.path.join(RUN_DIR, f'igca_best_{SLUG}.pt'))
SEC_PER_STEP = (time.time() - t0) / MAX_STEPS
print(f'best held-out {best["loss"]:.4f} at step {best["step"]} (gate {best["gate"]:.4f})')

## Evaluate the best checkpoint and plot the training curve

In [ ]:
import matplotlib.pyplot as plt
ckpt = torch.load(os.path.join(RUN_DIR, f'igca_best_{SLUG}.pt'))
bridge.modules.load_state_dict(ckpt['igca_state'])
OFF_FINAL, ON_FINAL = heldout(False), heldout(True)
print(f'held-out gate OFF {OFF_FINAL:.4f} | gate ON {ON_FINAL:.4f} | delta {OFF_FINAL - ON_FINAL:+.4f} nats')

w = max(1, len(loss_history) // 100)
plt.figure(figsize=(9, 4))
plt.plot(loss_history, alpha=0.3, label='per-step train loss')
if w > 1:
    ma = np.convolve(loss_history, np.ones(w) / w, mode='valid')
    plt.plot(range(w - 1, w - 1 + len(ma)), ma, lw=2, label=f'moving avg (w={w})')
plt.xlabel('step'); plt.ylabel('loss (nats)'); plt.title(f'IGCA on {SLUG} ({VARIANT})')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(os.path.join(RUN_DIR, 'loss_curve.png'), dpi=150, bbox_inches='tight'); plt.show()

# qualitative caption, gate off vs on
ex = heldout_ds[0]
bridge.set_modality(pipe.embed_images(vision_encoder, image_processor(images=ex['image'].convert('RGB'),
                                                                   return_tensors='pt').pixel_values))
prompt = tokenizer(pipe.PROMPT, return_tensors='pt').to(next(llm.parameters()).device)
llm.eval(); llm.config.use_cache = True
for name, ctx in [('gate OFF', bridge.disabled()), ('gate ON ', torch.no_grad())]:
    with ctx, torch.no_grad():
        out = llm.generate(**prompt, max_new_tokens=20, do_sample=False)
    print(f'[{name}]', tokenizer.decode(out[0], skip_special_tokens=True))
print('[truth  ]', ex['caption_0'])
llm.config.use_cache = False; llm.train()

## Run manifest and pre-registered decision (measured values only)

In [ ]:
delta = OFF_FINAL - ON_FINAL
verdict = (pipe.decision(STEP0_DIFF, best['gate'], delta) if VARIANT == 'gated'
           else f'UNGATED BASELINE: best held-out {ON_FINAL:.4f} (off {OFF_FINAL:.4f})')
manifest = igca.manifest_base(
    model_id=BASE_MODEL_ID, quantization=QUANT, vision_encoder=VISION_ENCODER_ID, variant=VARIANT,
    layer_path=bridge.layer_path, num_layers=bridge.num_layers, hidden_size=bridge.hidden,
    modality_dim=MODALITY_DIM, igca_indices=bridge.indices, n_heads=N_HEADS,
    trainable_params=N_TRAINABLE, frozen_params=FROZEN, step0_max_logit_diff=STEP0_DIFF,
    optimizer=OPT_NAME, objective=OBJECTIVE, l2_mu=L2_MU, lr_gate=LR_GATE, lr_weight=LR_WEIGHT,
    steps=MAX_STEPS, batch=BATCH_SIZE, seqlen=SEQLEN, seed=SEED,
    data={'dataset': pipe.FLICKR8K_ID, 'train_rows': len(train_ds), 'heldout_split': HELDOUT_SPLIT,
          'heldout_rows': len(heldout_ds), 'epochs': round(EPOCHS, 2)},
    train_loss_first=loss_history[0], train_loss_last=loss_history[-1],
    train_loss_last10=[round(x, 4) for x in loss_history[-10:]],
    heldout_gate_off=OFF_FINAL, heldout_step0_bridge_active=ON0, best_heldout_gate_on=ON_FINAL,
    best_step=best['step'], mean_gate_at_best=best['gate'], heldout_delta_nats=delta,
    heldout_delta_pct=100 * delta / OFF_FINAL, sec_per_step=round(SEC_PER_STEP, 3), decision=verdict)
path = log.manifest(manifest); log.close()
print('=' * 70)
for k in ['model_id', 'quantization', 'igca_indices', 'trainable_params', 'step0_max_logit_diff', 'data',
          'heldout_gate_off', 'best_heldout_gate_on', 'heldout_delta_nats', 'mean_gate_at_best', 'decision']:
    print(f'{k:>24}: {manifest[k]}')
print('manifest:', path)

## Keep the evidence
On Kaggle, click **Save Version**: the run directory (`run_manifest.json`, `loss_log.csv`,
`eval_log.csv`, `loss_curve.png`, best checkpoint) becomes a versioned dataset that anyone can audit.